# Resume Text Extract & Deep Cleaning

### Text extraction using pymudpf package

In [ ]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.5/22.5 MB 2.0 MB/s  0:00:11m0:00:0100:01


In [ ]:
import fitz
import re

def extract_resume_text(pdf_path):
    doc = fitz.open(pdf_path)
    result = []

    for page in doc:

        # Try standard text extraction (most reliable for many resumes)
        text = page.get_text("text", sort=True)
        if text and text.strip():
            result.append(text)
            continue

        # Try block extraction
        blocks = page.get_text("blocks")
        if blocks:
            block_text = "\n".join(b[4] for b in blocks if len(b) > 4)
            if block_text.strip():
                result.append(block_text)
                continue

        # Try rawdict spans (used in our original code)
        raw = page.get_text("rawdict")
        if raw and "blocks" in raw:
            for block in raw["blocks"]:
                if "lines" in block:
                    for line in block["lines"]:
                        spans = [span["text"] for span in line["spans"]]
                        if spans:
                            result.append(" ".join(spans))

        # Try XML mode (captures weird PDF encodings)
        xml = page.get_text("xml")
        if xml and xml.strip():
            result.append(xml)

        # Last fallback → HTML mode
        html = page.get_text("html")
        if html and html.strip():
            result.append(html)

    doc.close()

    # Normalize bullets & whitespace
    text = "\n".join(result)
    text = re.sub(r"[•●▪■◆▶►]", "-", text)
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text)

    return text


In [56]:
pdf_path = "Resume/Resume Example 1.pdf"

text = extract_resume_text(pdf_path)

print("\n===== EXTRACTED TEXT PREVIEW =====\n")
print(text)  # print first 1500 chars
print("\n===================================\n")



===== EXTRACTED TEXT PREVIEW =====

                  David J. Frederickson
                 510.378.6911 │ david.j.frederickson@gmail.com | www.linkedin.com/in/davidfrederickson
                            Summary
-   Results-driven professional with extensive experience in procurement, supply chain, logistics, and project management.
-   Proven ability to identify and capitalize on opportunities to increase operational efficiency while minimizing expenses.
-   Exceptional project management abilities combined with strong analytical, organizational, and communication skills.

 -   Vendor & Supplier Relations              -   Strategic Negotiations                      -   Legal contract writing
 -   Data Collection & Analysis                -  Expense & Cost Reduction                 -   Inventory Control (ERP)
 -   RF(X)                                   -  Advanced Microsoft Excel                  -   Process Improvement

                                    Professional Experience


### Text Deep Cleaning

In [57]:
import re
import unicodedata

def clean_resume_text(text):

    text = unicodedata.normalize("NFKD", text)
    text = re.sub(r"[\u200b\u200c\u200d\u2060\ufeff]", "", text)

    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"\+?\d[\d\-\s\(\)]{7,}\d", " ", text)
    text = re.sub(r"(https?:\/\/\S+|www\.\S+)", " ", text)
    text = re.sub(r"(linkedin|github)\S*", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\b\d{5}(?:-\d{4})?\b", " ", text)

    pii_words = ["email", "phone", "linkedin", "github", "contact", "address"]
    filtered = []
    for line in text.split("\n"):
        if not line.strip().lower().startswith(tuple(pii_words)):
            filtered.append(line)
    text = "\n".join(filtered)

    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"&[a-z]+;", " ", text)

    text = re.sub(r"^[•●▪■◆▶►▸⦿⦾]\s*", "- ", text, flags=re.MULTILINE)
    text = re.sub(r"^-(\S)", r"- \1", text, flags=re.MULTILINE)

    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    text = text.replace("–", "-").replace("—", "-")

    text = text.replace("\t", " ")
    text = re.sub(r" {2,}", " ", text)

    # ---------------------------------------
    # Remove everything before the first real section header
    # ---------------------------------------
    section_headers = [
    # Skills / Expertise
    "skills", "technical skills", "areas of expertise", "expertise",
    "core competencies", "competencies", "strengths",
    "tools", "certifications",

    # Experience
    "professional experience", "experience", "work experience",
    "employment history",

    # Education
    "education", "academic background", "academic history",

    # Projects
    "projects", "project experience", "relevant project experience",
    "academic projects",

    # Summary / Profile
    "summary", "professional summary", "profile",
    "about me", "overview", "objective"
]

    cleaned_lines = []
    found = False
    for line in text.split("\n"):
        stripped = line.strip().lower()
        if any(stripped.startswith(h) for h in section_headers):
            found = True
        if found:
            cleaned_lines.append(line)

    # ---------------------------------------
    # Insert EXACTLY 1 blank line before each section header
    # ---------------------------------------
    final_lines = []
    for i, line in enumerate(cleaned_lines):
        stripped = line.strip().lower()
        if any(stripped.startswith(h) for h in section_headers):
            # Not for first header
            if len(final_lines) > 0 and final_lines[-1].strip() != "":
                final_lines.append("")
        final_lines.append(line)

    cleaned_lines = final_lines

    # ---------------------------------------
    # Collapse extra blank lines, allow max 1
    # ---------------------------------------
    result = []
    blank = False
    for line in cleaned_lines:
        if line.strip() == "":
            if not blank:
                result.append("")
            blank = True
        else:
            result.append(line.rstrip())
            blank = False

    text = "\n".join(result)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


In [58]:
cleaned_text = clean_resume_text(text)
print("\n===== CLEANED TEXT PREVIEW =====\n")
print(cleaned_text)  # print first 1500 chars
print("\n=================================\n")


===== CLEANED TEXT PREVIEW =====

Summary
- Results-driven professional with extensive experience in procurement, supply chain, logistics, and project management.
- Proven ability to identify and capitalize on opportunities to increase operational efficiency while minimizing expenses.
- Exceptional project management abilities combined with strong analytical, organizational, and communication skills.

 - Vendor & Supplier Relations - Strategic Negotiations - Legal contract writing
 - Data Collection & Analysis - Expense & Cost Reduction - Inventory Control (ERP)
 - RF(X) - Advanced Microsoft Excel - Process Improvement

 Professional Experience

Apple Inc., Cupertino, CA 2017 - 2019
Procurement Manager
Serves as key member of Preservation Sourcing team - Managing suppliers and tracking daily output vs store demand, logistics,
quality issues, and P.O. and invoice management.
- Plan and present annual supplier reviews with worldwide service providers and manufacturing partners
- Analyze

### NER Based Skill Extraction

For skill extraction, we built a custom Skill-NER model using spaCy's PhraseMatcher and a curated dictionary of 450+ skills. This allowed us to define our own ‘SKILL’ entity type without requiring a large annotated dataset. In addition, spaCy’s statistical NER (ORG, GPE, DATE) was used to extract metadata from resumes and job postings. This combination of rule-based and statistical NER forms a hybrid NER approach aligned with Module 4 of the course.

### Custom SKILL dictionary

In [59]:
MASTER_SKILL_LIST = [

    # ------------------------------
    # TECHNICAL / DATA SKILLS
    # ------------------------------
    "python", "r", "java", "javascript", "typescript",
    "c++", "c#", "scala", "go", "matlab",
    "bash", "shell scripting",

    # Data Analytics
    "sql", "nosql", "postgresql", "mysql", "oracle", "sqlite",
    "mongodb", "snowflake", "redshift", "bigquery", "azure sql",
    "data analysis", "data analytics", "statistical analysis",

    # Data Tools
    "pandas", "numpy", "scipy", "matplotlib", "seaborn",
    "plotly", "pyspark", "spark", "hadoop", "hive", "mapreduce",

    # Machine Learning
    "machine learning", "deep learning", "neural networks",
    "logistic regression", "linear regression", "random forest",
    "xgboost", "lightgbm", "catboost",
    "svm", "knn", "decision trees", "pca", "kmeans",
    "gradient boosting", "model tuning", "feature engineering",

    # NLP
    "nlp", "natural language processing", "topic modeling",
    "lda", "lsa", "keyword extraction",
    "named entity recognition", "text classification",
    "sentiment analysis", "embeddings", "bert", "word2vec",

    # Cloud
    "aws", "azure", "gcp", "docker", "kubernetes",
    "lambda", "ec2", "s3", "athena", "dynamodb",
    "databricks", "airflow", "cloud functions",

    # BI Tools
    "tableau", "power bi", "metabase", "looker", "qlik",
    "data visualization", "dashboard development",

    # ETL / Pipelines
    "etl", "elt", "data pipeline", "data ingestion",
    "data cleaning", "data transformation", "data integration",

    # Version Control & DevOps
    "git", "github", "gitlab", "bitbucket",
    "ci/cd", "jenkins",

    # Enterprise Tools
    "sap", "sap erp", "salesforce", "salesforce crm",
    "hubspot", "hubspot crm", "airtable", "jira", "confluence", "notion",

    # ------------------------------
    # BUSINESS & ANALYTICS SKILLS
    # ------------------------------
    "business analysis", "requirements gathering",
    "market research", "competitive analysis",
    "financial analysis", "risk analysis", "cost analysis",
    "forecasting", "trend analysis", "variance analysis",
    "p&l management", "strategic planning",
    "business modeling", "stakeholder management",
    "reporting", "presentation development",
    "process improvement", "process optimization",
    "root cause analysis", "gap analysis",
    "workflow automation", "operational efficiency",
    "kpi analysis", "performance analysis",
    "customer segmentation", "persona development",
    "data-driven decision making",

    # Consulting skills
    "problem solving", "insights synthesis",
    "client communication", "proposal writing",
    "project scoping", "roadmap planning",
    "change management", "cross-functional collaboration",

    # ------------------------------
    # MARKETING / SALES / REVOPS SKILLS
    # ------------------------------
    "crm management", "lead generation", "pipeline management",
    "sales operations", "sales strategy", "sales forecasting",
    "revenue operations", "revops", "gtm strategy",
    "go-to-market", "account management",
    "client success", "customer retention",

    # Marketing
    "digital marketing", "content marketing",
    "seo", "sem", "ppc", "email marketing",
    "campaign optimization", "social media analytics",

    # Marketing tools
    "marketing automation", "google analytics",
    "google ads", "mailchimp", "marketo",
    "outreach", "gong", "zoominfo",

    # RevOps Processes
    "validation rules", "crm integrations",
    "funnel analysis", "data stamping",

    # ------------------------------
    # PRODUCT SKILLS
    # ------------------------------
    "product management", "product analytics",
    "a/b testing", "experiment design",
    "feature prioritization", "user research", "ux research",
    "user stories", "agile", "scrum", "kanban",
    "roadmap development", "user journey mapping",
    "requirements documentation",
    "market sizing", "competitive positioning",

    # ------------------------------
    # FINANCE & OPERATIONS SKILLS
    # ------------------------------
    "fp&a", "financial modeling", "budgeting",
    "scenario analysis", "invoice processing",
    "billing operations", "revenue analysis",
    "cost optimization",

    # Operations & Supply Chain
    "supply chain management", "inventory management",
    "logistics", "procurement", "vendor management",
    "operations management", "kpi reporting",

    # ------------------------------
    # SOFT SKILLS
    # ------------------------------
    "communication", "leadership", "teamwork",
    "collaboration", "critical thinking", "problem solving",
    "adaptability", "time management",
    "presentation skills", "negotiation",
    "public speaking", "project management",
    "detail oriented", "strategic thinking",
    "multitasking", "analytical thinking",
    "decision making", "organization skills"
]

MASTER_SKILL_LIST = list(set(MASTER_SKILL_LIST))


In [60]:
import spacy
from spacy.matcher import PhraseMatcher

nlp = spacy.load("en_core_web_sm")


def build_skill_ner(skill_list):
    """
    Builds a spaCy PhraseMatcher for custom skill extraction.
    """
    matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

    # Create spaCy Doc patterns for each skill phrase
    patterns = [nlp.make_doc(skill) for skill in skill_list]
    matcher.add("SKILL", patterns)

    return matcher

skill_matcher = build_skill_ner(MASTER_SKILL_LIST)

In [61]:
def extract_skill_entities(text):
    """
    Extracts skill entities from text using the SKILL PhraseMatcher.
    Returns a unique list of skills (lowercased).
    """
    doc = nlp(text)
    matches = skill_matcher(doc)

    skills_found = set()

    for match_id, start, end in matches:
        span = doc[start:end]
        skills_found.add(span.text.lower())

    return sorted(list(skills_found))


In [63]:
skills_in_resume = extract_skill_entities(cleaned_text)
print("Extracted Skills:")
print(skills_in_resume)

Extracted Skills:
['communication', 'forecasting', 'leadership', 'logistics', 'operational efficiency', 'oracle', 'process improvement', 'procurement', 'project management', 'reporting', 'sales operations']


### Custom SKILL recognizer using PhraseMatcher

In [79]:
import spacy
spacy.displacy.render.__globals__["is_in_jupyter"] = lambda: False


class HTMLVisualizer:
    def __init__(self, html):
        self.html = html

    def _repr_html_(self):
        return self.html


def visualize_skill_entities(text):
    doc = nlp(text)
    matches = skill_matcher(doc)

    ents = []
    for match_id, start, end in matches:
        span = doc[start:end]
        ents.append({
            "start": span.start_char,
            "end": span.end_char,
            "label": "SKILL"
        })

    doc_dict = {"text": text, "ents": ents}

    html = displacy.render(doc_dict, style="ent", manual=True)

    return HTMLVisualizer(html)

visualize_skill_entities(cleaned_text)

In [81]:
from spacy import displacy
from IPython.display import HTML

def visualize_all_entities(text):
    """
    Visualize BOTH:
    - spaCy NER (ORG, DATE, GPE, etc.)
    - custom SKILL entities
    """
    doc = nlp(text)

    # Default spaCy entities
    ents = [{
        "start": ent.start_char,
        "end": ent.end_char,
        "label": ent.label_
    } for ent in doc.ents]

    # Add custom SKILL entities
    matches = skill_matcher(doc)
    for match_id, start, end in matches:
        span = doc[start:end]
        ents.append({
            "start": span.start_char,
            "end": span.end_char,
            "label": "SKILL"
        })

    # Sort entities by start index
    ents = sorted(ents, key=lambda x: x["start"])

    # Build manual entity rendering dict
    doc_dict = {"text": text, "ents": ents}

    # --- SAFE HTML rendering (spaCy will NOT import display now) ---
    html = displacy.render(doc_dict, style="ent", manual=True)

    return HTML(html)

visualize_all_entities(cleaned_text)